# Recursion, Memoization, and Dynamic Programming (DP)

## 🌱 Learning Goals

By the end of this notebook, you should be able to:

- Understand recursive thinking  
- Identify overlapping subproblems  
- Apply memoization for efficiency  
- Understand the core idea behind dynamic programming  
- Distinguish dynamic programming from divide and conquer approaches  


### _Code Setup_

```bash
pip install rich ipywidgets ipykernel
```

In [1]:
from rich import get_console, print

# Set up the console for rich output, avoid issues in Jupyter notebooks.
console = get_console()
console.is_jupyter = False

In [2]:
import ipywidgets as widgets
from IPython.display import display

box_layout = widgets.Layout(width="50%", border="1px solid lightgray", padding="10px")

In [3]:
def create_side_by_side():
    left = widgets.Output(layout=box_layout)
    right = widgets.Output(layout=box_layout)
    container = widgets.HBox([left, right], layout=widgets.Layout(gap="20px"))
    display(container)
    return left, right

## 1) Recursion

Recursion is a problem solving approach where a function calls itself to solve smaller versions of the same problem.

A recursive function contains:

- A base case  
- A recursive case

📘 Check out *Data Structures and Algorithms with Python*, p. 74, Section 3.4 for _how to write a recursive function_.

The free online version of the book is available under "Main Learning Materials" on Spaces.

### 1.1) Recursive Example: Reverse Number Triangle

#### Goal

Print:

```text
12345
1234
123
12
1
```


#### Iterative Version

In [7]:
def it_reverse_triangle(n):
    for i in range(n, 0, -1):
        for j in range(1, i + 1):
            print(j, end="")
        print()

#### Recursive Version

In [8]:
def rec_reverse_triangle(n):
    if n == 0:
        return
    for j in range(1, n + 1):
        print(j, end="")
    print()
    rec_reverse_triangle(n - 1)

#### Comparison

In [9]:
left, right = create_side_by_side()

with left:
    left.clear_output()
    print("<== Iterative version ==>")
    it_reverse_triangle(5)

with right:
    right.clear_output()
    print("<== Recursive version ==>")
    rec_reverse_triangle(5)

### 1.2) Understanding the Recursive Structure

#### Base Case

```python
if n == 0:
    return
```

Stops the recursion.

#### Recursive Case

```python
rec_reverse_triangle(n - 1)
```

Solves a smaller version of the same problem.


## 2) Overlapping Subproblems

Some recursive problems repeatedly solve the same smaller subproblems which creates unnecessary computation.

This concept is called overlapping subproblems.


### 2.1) Example: Grid Path Counting

Imagine a grid.

You start from the top left corner and want to reach the bottom right corner.

You can only move:
- Right  
- Down


#### Recursive Solution

In [10]:
def count_paths(rows, cols):
    if rows == 1 or cols == 1:
        return 1

    return count_paths(rows - 1, cols) + count_paths(rows, cols - 1)


print(count_paths(3, 3))

6


#### **Why is this inefficient?**

The same subproblems are recomputed many times.

Example:

```text
count_paths(3, 3)

calls:

count_paths(2, 3)
count_paths(3, 2)

both later call:

count_paths(2, 2)
```

This causes unnecessary runtime growth.


### 2.2) Memoization for Efficiency

> Memoization is an optimization technique used to speed up computer programs.

Memoization stores previously computed results.

Instead of recomputing a subproblem, we **reuse** the stored answer.


#### Memoized Grid Path Solution

In [11]:
def count_paths_memo(rows, cols, memo={}):

    if (rows, cols) in memo:
        return memo[(rows, cols)]

    if rows == 1 or cols == 1:
        return 1

    memo[(rows, cols)] = count_paths_memo(rows - 1, cols, memo) + count_paths_memo(
        rows, cols - 1, memo
    )

    return memo[(rows, cols)]


print(count_paths_memo(15, 15))

40116600


#### ⭐ Python Built-in Memoization

Python provides a built-in memoization utility through:
```python
from functools import lru_cache
```
This automatically stores previously computed results.

LRU = Least Recently Used


In [12]:
from functools import lru_cache


@lru_cache(maxsize=256)
def count_paths_cached(rows, cols):

    if rows == 1 or cols == 1:
        return 1

    return count_paths_cached(rows - 1, cols) + count_paths_cached(rows, cols - 1)


print(count_paths_cached(15, 15))

40116600


⚠️ **Limitations:**
- Inputs must be hashable.
- Performance impact: A small cache increases cache misses, requiring frequent recomputation, while an excessively large cache can exhaust system memory.
    - Cache size may grow large for huge problems.
    - Older cached values may be removed when the cache becomes full.

✨ There is also `cache()` by `functools`, check it out yourself [here](https://docs.python.org/3/library/functools.html).

### 2.3) Timing Comparison

In [13]:
import time

start = time.perf_counter()
count_paths(15, 15)
end = time.perf_counter()

print(f"Recursive version: {end - start:.4f} seconds")

Recursive version: 4.8449 seconds


In [18]:
start = time.perf_counter()
count_paths_memo(15, 15)
end = time.perf_counter()

print(f"Memoized version: {end - start:.6f} seconds")

Memoized version: 0.000036 seconds


In [19]:
start = time.perf_counter()
count_paths_cached(15, 15)
end = time.perf_counter()

print(f"Cached version: {end - start:.6f} seconds")

Cached version: 0.000033 seconds


## 3) Dynamic Programming

Dynamic Programming, DP, is an optimization technique for problems with:

- Overlapping subproblems  
- Optimal substructure

Instead of recomputing results:
1. we store them  
2. and reuse them later


📘 Check out *Introduction to Algorithms* by Thomas H. Cormen et al., Chapter 15 "Dynamic Programming", for more information on this topic.

The free online version of the book is available under "Main Learning Materials" on Spaces.

### 3.1) Top Down vs Bottom Up

#### Top Down DP
- Recursive  
- Usually uses memoization  

Example:
```python
count_paths_memo()
```

#### Bottom Up DP
- Iterative  
- Builds solutions from smaller states upward


| Top Down | Bottom Up |
|---|---|
| Recursive | Iterative _(loops)_ |
| **Storage Method:** Memoization | **Storage Method:** DP tables/arrays |
| Starts from the final problem | Starts from the smallest subproblems |
| Computes only needed | Computes all states systematically |
| Easier to write and reason about | More efficient in practice |
| Has recursion overhead | Avoids recursion overhead |
| Can hit recursion depth limits | No recursion depth issue |

#### 3.1.1) Bottom Up Grid Path DP

In [20]:
def count_paths_dp(rows, cols):

    dp = [[0] * cols for _ in range(rows)]

    for i in range(rows):
        dp[i][0] = 1

    for j in range(cols):
        dp[0][j] = 1

    for i in range(1, rows):
        for j in range(1, cols):
            dp[i][j] = dp[i - 1][j] + dp[i][j - 1]

    return dp[-1][-1]


print(count_paths_dp(15, 15))

40116600


### ⭐ DP: General Recipe

When solving DP problems:

1. Define the state  
2. Define the recurrence relation  
3. Identify base cases  
4. Decide how to store results  
5. Build the solution

⚠️ **Dynamic programming improves runtime by trading additional memory for speed.**

## 4) Divide and Conquer vs Dynamic Programming

| Divide and Conquer | Dynamic Programming |
|---|---|
| Splits problem into independent subproblems | Subproblems overlap |
| Usually does not reuse computations | Reuses stored results |
| Examples: Merge Sort, Quick Sort | Examples: Memoization, shortest paths |
| Focus on decomposition | Focus on avoiding repeated work |


## 5) Space Complexity

1. **Recursion Stack Space:**

    This comes from function calls themselves.
    
    Every recursive call adds a new frame to the call stack.
    
    Therefore, recursion alone already adds extra memory usage. → O(depth of recursion)


2. **Memoization / DP Storage Space:**

    This comes from storing previous results. → O(number of stored states)

    Examples:
    - Dictionary cache
    - DP table
    - Arrays

## 6) Take-home Lessons

- Recursion solves problems using smaller versions of the same problem  
- Some recursive problems recompute identical subproblems  
- Memoization avoids repeated work  
- Dynamic programming (DP) stores and reuses previous results  
- DP is especially useful for optimization and structured data problems  
- DP differs from divide and conquer because subproblems overlap
